# 03_Graph_Analytics

Select BDA Local (Python 3.11), or the project .venv Python environment, in VS Code. Run 01, 02, and 03 sequentially. All CSV inputs, Parquet outputs, reports, and staging files use the local project root (or BDA_PROJECT_DIR). No Colab connection, uploads, exports, or Drive mounting are used. The local prototype uses four Spark threads. Missing label dates are excluded; delay defaults to timestamp.


In [1]:
import os
import sys
import json
import subprocess
import tempfile
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

if os.name != "nt":
    raise RuntimeError("Select the local Windows BDA Local kernel in VS Code. This notebook does not use Colab.")
# Keep all reads/writes local to this checkout; override on another device if needed.
project_hint = os.environ.get("BDA_PROJECT_DIR")
project_cwd = Path.cwd().resolve()
base_dir = (Path(project_hint).expanduser().resolve() if project_hint else
            project_cwd.parent if project_cwd.name == "notebooks" else project_cwd)
if not (base_dir / "notebooks").is_dir():
    raise FileNotFoundError("Open the project root/notebooks directory or set BDA_PROJECT_DIR to the local checkout.")
raw_dir = base_dir / "datasets_raw"
parquet_dir = base_dir / "datasets_parquet"
output_dir = base_dir / "outputs_ml_graph"
runtime_dir = base_dir / ".runtime"
temp_dir = runtime_dir / "tmp"
temp_dir.mkdir(parents=True, exist_ok=True)
for name in ["TMP", "TEMP", "TMPDIR"]:
    os.environ[name] = str(temp_dir)
tempfile.tempdir = str(temp_dir)
java_candidates = sorted(p for p in runtime_dir.glob("jdk-*") if (p / "bin/java.exe").exists())
if not java_candidates:
    raise FileNotFoundError("The project-local Java runtime is missing from .runtime/jdk-*. Complete local setup first.")
os.environ["JAVA_HOME"] = str(java_candidates[-1])
os.environ["HADOOP_HOME"] = str(runtime_dir / "hadoop")
os.environ["SPARK_HOME"] = str(runtime_dir / "spark-home")
os.environ["PATH"] = (str(java_candidates[-1] / "bin") + os.pathsep +
                      str(runtime_dir / "hadoop/bin") + os.pathsep + os.environ["PATH"])
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
required_csvs = [
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]
missing = [name for name in required_csvs if not (raw_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing local CSVs in {raw_dir}: {missing}")
output_dir.mkdir(parents=True, exist_ok=True)
DELAY_DEFINITION = os.environ.get("BDA_DELAY_DEFINITION", "timestamp")
if DELAY_DEFINITION not in {"timestamp", "calendar_date"}:
    raise ValueError("BDA_DELAY_DEFINITION must be timestamp or calendar_date")

def ensure_package(name, required_version=None):
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = None
    if installed is None or (required_version and installed != required_version):
        spec = f"{name}=={required_version}" if required_version else name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

print(f"BDA LOCAL WINDOWS | Python: {sys.executable}")
print(f"All project data and outputs: {base_dir}")
print(f"Delay definition: {DELAY_DEFINITION}")
ensure_package("pandas")
ensure_package("networkx", "3.4.2")
ensure_package("scipy")


BDA LOCAL WINDOWS | Python: d:\Private Project\bda\BDA_Olist_Project\.venv\Scripts\python.exe
All project data and outputs: D:\Private Project\bda\BDA_Olist_Project
Delay definition: timestamp


In [2]:
import pandas as pd
import networkx as nx

orders = pd.read_csv(raw_dir / "olist_orders_dataset.csv")
items = pd.read_csv(raw_dir / "olist_order_items_dataset.csv")
customers = pd.read_csv(raw_dir / "olist_customers_dataset.csv")
sellers = pd.read_csv(raw_dir / "olist_sellers_dataset.csv")
for df, keys, name in [(orders, ["order_id"], "orders"), (items, ["order_id", "order_item_id"], "items"),
                       (customers, ["customer_id"], "customers"), (sellers, ["seller_id"], "sellers")]:
    if df[keys].isna().any().any() or df.duplicated(keys).any():
        raise ValueError(f"Invalid/duplicate keys: {name}")
delivered = orders.loc[orders.order_status.eq("delivered")].copy()
for column in ["order_delivered_customer_date", "order_estimated_delivery_date"]:
    delivered[column] = pd.to_datetime(delivered[column], errors="raise")
missing_dates = int(delivered[["order_delivered_customer_date", "order_estimated_delivery_date"]].isna().any(axis=1).sum())
delivered = delivered.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"])
actual, estimate = delivered.order_delivered_customer_date, delivered.order_estimated_delivery_date
timestamp_delay = actual > estimate
calendar_delay = actual.dt.normalize() > estimate.dt.normalize()
delivered["is_delayed"] = (timestamp_delay if DELAY_DEFINITION == "timestamp" else calendar_delay).astype(int)

# Same representative item rule as notebook 02; seller/product come from one actual row.
primary = items.sort_values(["order_id", "order_item_id"]).drop_duplicates("order_id")
primary = primary[["order_id", "seller_id"]].rename(columns={"seller_id": "primary_seller_id"})
items_agg = items.groupby("order_id", as_index=False).agg(
    total_freight=("freight_value", "sum"), seller_count=("seller_id", "nunique"))
items_agg = items_agg.merge(primary, on="order_id", validate="one_to_one")
if not delivered.order_id.isin(items_agg.order_id).all():
    raise ValueError("Delivered orders without items")
merged = delivered.merge(items_agg, on="order_id", validate="one_to_one")
merged = merged.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left", validate="many_to_one")
merged = merged.merge(sellers[["seller_id", "seller_state"]], left_on="primary_seller_id", right_on="seller_id", how="left", validate="many_to_one")
if merged[["seller_state", "customer_state"]].isna().any().any():
    raise ValueError("Missing seller/customer state or orphan key")
routes_pd = merged.groupby(["seller_state", "customer_state"], as_index=False).agg(
    transaction_volume=("is_delayed", "size"), delayed_orders=("is_delayed", "sum"),
    route_delay_rate_pct=("is_delayed", lambda x: round(x.mean() * 100, 2)),
    avg_freight_value=("total_freight", lambda x: round(x.mean(), 2)))
routes_pd = routes_pd.sort_values(["transaction_volume", "seller_state", "customer_state"], ascending=[False, True, True])
G = nx.DiGraph()
for row in routes_pd.itertuples(index=False):
    G.add_edge(row.seller_state, row.customer_state, weight=float(row.transaction_volume),
               delay_rate=float(row.route_delay_rate_pct), avg_freight=float(row.avg_freight_value))
if sum(dict(G.out_degree(weight="weight")).values()) != len(merged):
    raise ValueError("Graph volume does not reconcile with order count")
routes_pd.to_csv(output_dir / "seller_customer_routes_report.csv", index=False)
graph_audit = {
    "delay_definition": DELAY_DEFINITION, "primary_item_rule": "minimum_order_item_id",
    "excluded_missing_label_dates": missing_dates, "graph_orders": len(merged),
    "delayed_orders": int(merged.is_delayed.sum()),
    "timestamp_delayed_orders": int(timestamp_delay.sum()),
    "calendar_date_delayed_orders": int(calendar_delay.sum()),
    "multi_seller_orders": int(merged.seller_count.gt(1).sum()),
    "nodes": G.number_of_nodes(), "routes": G.number_of_edges(),
}
feature_audit_path = output_dir / "feature_preparation_audit.json"
if feature_audit_path.exists():
    feature_audit = json.loads(feature_audit_path.read_text(encoding="utf-8"))
    for graph_key, feature_key in [("delay_definition", "delay_definition"), ("primary_item_rule", "primary_item_rule"),
                                  ("graph_orders", "feature_orders"), ("delayed_orders", "delayed_orders")]:
        if graph_audit[graph_key] != feature_audit[feature_key]:
            raise ValueError(f"Notebook 02/03 mismatch: {graph_key}. Rerun both with the same inputs/configuration.")
else:
    print("Notebook 02 audit not found: cross-notebook reconciliation skipped.")
(output_dir / "graph_preparation_audit.json").write_text(json.dumps(graph_audit, indent=2), encoding="utf-8")
print(json.dumps(graph_audit, indent=2))


{
  "delay_definition": "timestamp",
  "primary_item_rule": "minimum_order_item_id",
  "excluded_missing_label_dates": 8,
  "graph_orders": 96470,
  "delayed_orders": 7826,
  "timestamp_delayed_orders": 7826,
  "calendar_date_delayed_orders": 6534,
  "multi_seller_orders": 1275,
  "nodes": 27,
  "routes": 409
}


In [3]:
out_strength = dict(G.out_degree(weight="weight"))
in_strength = dict(G.in_degree(weight="weight"))
pagerank = nx.pagerank(G, weight="weight")
# Topological shortest paths: volume is not a physical distance/cost.
# These paths do not establish actual package transit through intermediate states.
betweenness = nx.betweenness_centrality(G, weight=None)
nodes_df = pd.DataFrame([{
    "State": node, "Out_Strength_Orders": int(out_strength[node]),
    "In_Strength_Orders": int(in_strength[node]), "PageRank_Score": pagerank[node],
    "Topological_Betweenness": betweenness[node],
} for node in sorted(G.nodes())]).sort_values(["Out_Strength_Orders", "State"], ascending=[False, True])
nodes_df.to_csv(output_dir / "graph_network_metrics.csv", index=False)
top_seller = nodes_df.iloc[0]
top_market = nodes_df.sort_values(["In_Strength_Orders", "State"], ascending=[False, True]).iloc[0]
filtered_routes = routes_pd.loc[routes_pd.transaction_volume > 500]
print(f"Seller hub: {top_seller.State}, {int(top_seller.Out_Strength_Orders):,} orders")
print(f"Market sink: {top_market.State}, {int(top_market.In_Strength_Orders):,} orders")
if filtered_routes.empty:
    print("No route exceeds 500 orders; no high-volume bottleneck ranking produced.")
else:
    bottleneck = filtered_routes.sort_values(
        ["route_delay_rate_pct", "transaction_volume", "seller_state", "customer_state"],
        ascending=[False, False, True, True]).iloc[0]
    print(f"Highest delay rate among routes >500 orders: {bottleneck.seller_state} -> {bottleneck.customer_state}; "
          f"{int(bottleneck.transaction_volume):,} orders; {bottleneck.route_delay_rate_pct:.2f}%")


Seller hub: SP, 68,415 orders
Market sink: SP, 40,494 orders
Highest delay rate among routes >500 orders: SP -> RJ; 8,158 orders; 15.49%
